# Data Model Build — AgentOps Cost & Governance

Builds a **Power BI–ready star schema** in the `Observability` lakehouse (`analytics` schema) from the raw
sources explored in `Data_Exploration`:

| Source | Feeds |
|---|---|
| `Files/costs/` (FOCUS export) | `fact_cost` |
| `Files/am-appdependencies/` (gen_ai spans) | `fact_genai_calls`, `dim_agent`, `dim_model` |
| `Files/metadata/` (Resource Graph) | `dim_resource` |
| generated | `dim_date` |

**Tables produced (minimized):**
- Dimensions: `dim_date`, `dim_resource`, `dim_agent`, `dim_model`
- Facts: `fact_cost`, `fact_genai_calls`
- Reporting facts: `fact_token_consumption` (report page: token usage), `fact_conversation_steps` (report page: conversation explorer)

**Relationships for Power BI (single-direction, dim -> fact):**
- `dim_date[date_key]` -> `fact_cost[charge_date]`, `fact_genai_calls[date_key]`, `fact_token_consumption[timeday]`, `fact_conversation_steps[date_key]`
- `dim_resource[resource_id]` -> `fact_cost[resource_id]`
- `dim_agent[agent_id]` -> `fact_genai_calls[agent_id]`, `fact_token_consumption[agent_id]`, `fact_conversation_steps[agent_id]`
- `dim_model[model_name]` -> `fact_genai_calls[response_model]`, `fact_token_consumption[response_model]`, `fact_conversation_steps[response_model]`

> User identity (`enduser_id`/`enduser_mail`) and feedback (`gen_ai.tracing.feedback`) are intentionally **not** implemented (not present in the exported dataset).


## 1. Setup & helpers

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType
from pyspark.sql.window import Window
from datetime import date

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.parquet.mergeSchema", "true")

SCHEMA = "analytics"


def write_table(df, name):
    """Overwrite a Delta table under Tables/analytics/ and report the row count."""
    path = f"Tables/{SCHEMA}/{name}"
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path))
    n = spark.read.format("delta").load(path).count()
    print(f"  wrote {SCHEMA}.{name}: {n} rows")


def has_struct_field(df, struct_col_name, field_name):
    field = next((f for f in df.schema.fields if f.name == struct_col_name), None)
    if field is None or not isinstance(field.dataType, StructType):
        return False
    return field_name in [f.name for f in field.dataType.fields]


def sc(df, struct_col_name, field_name, data_type="string"):
    """Safe access to a nested (dotted) struct field; NULL if absent."""
    if has_struct_field(df, struct_col_name, field_name):
        return F.col(f"{struct_col_name}.`{field_name}`").cast(data_type)
    return F.lit(None).cast(data_type)


def fc(df, name, data_type="string"):
    """Safe access to a flat column whose name may contain dots; NULL if absent."""
    if name in df.columns:
        return F.col(f"`{name}`").cast(data_type)
    return F.lit(None).cast(data_type)


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 3, Finished, Available, Finished, False)

## 2. Load raw sources

In [5]:
df_costs_raw = (
    spark.read.format("parquet")
    .option("mergeSchema", "true")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.parquet")
    .load("Files/costs/")
)

df_appdeps_raw = (
    spark.read.format("json")
    .option("mergeSchema", "true")
    .option("recursiveFileLookup", "true")
    .load("Files/appdependencies/")
)

df_metadata_raw = (
    spark.read.format("parquet")
    .option("mergeSchema", "true")
    .option("recursiveFileLookup", "true")
    .load("Files/metadata/")
    .withColumn("_source_file", F.input_file_name())
)
df_metadata_raw = df_metadata_raw.toDF(*[c.replace(" ", "") for c in df_metadata_raw.columns])

print(f"costs rows:    {df_costs_raw.count()}")
print(f"appdeps rows:  {df_appdeps_raw.count()}")
print(f"metadata rows: {df_metadata_raw.count()}")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 7, Finished, Available, Finished, False)

costs rows:    11259
appdeps rows:  485
metadata rows: 264


## 3. `fact_cost`
One row per FOCUS daily charge line. Grain: date x resource x meter. Includes flags for AI services and
the `Foundry Models` meter so per-model billed spend can be isolated.

In [6]:
fact_cost = (
    df_costs_raw
    .withColumn("charge_date", F.to_date("ChargePeriodStart"))
    .select(
        F.col("charge_date"),
        F.to_date("BillingPeriodStart").alias("billing_period_start"),
        F.to_date("BillingPeriodEnd").alias("billing_period_end"),
        fc(df_costs_raw, "ResourceId").alias("resource_id"),
        fc(df_costs_raw, "ResourceName").alias("resource_name"),
        fc(df_costs_raw, "ResourceType").alias("resource_type"),
        fc(df_costs_raw, "ServiceName").alias("service_name"),
        fc(df_costs_raw, "ServiceCategory").alias("service_category"),
        fc(df_costs_raw, "RegionId").alias("region_id"),
        fc(df_costs_raw, "RegionName").alias("region_name"),
        fc(df_costs_raw, "SubAccountId").alias("subscription_id"),
        fc(df_costs_raw, "SubAccountName").alias("subscription_name"),
        fc(df_costs_raw, "x_ResourceGroupName").alias("resource_group"),
        fc(df_costs_raw, "SkuId").alias("sku_id"),
        fc(df_costs_raw, "x_SkuMeterCategory").alias("meter_category"),
        fc(df_costs_raw, "x_SkuMeterSubcategory").alias("meter_subcategory"),
        fc(df_costs_raw, "x_SkuMeterName").alias("meter_name"),
        fc(df_costs_raw, "ChargeCategory").alias("charge_category"),
        fc(df_costs_raw, "ChargeDescription").alias("charge_description"),
        fc(df_costs_raw, "PricingCategory").alias("pricing_category"),
        fc(df_costs_raw, "CommitmentDiscountType").alias("commitment_discount_type"),
        fc(df_costs_raw, "ConsumedUnit").alias("consumed_unit"),
        fc(df_costs_raw, "PricingUnit").alias("pricing_unit"),
        fc(df_costs_raw, "Tags").alias("tags"),
        fc(df_costs_raw, "BilledCost", "double").alias("billed_cost"),
        fc(df_costs_raw, "EffectiveCost", "double").alias("effective_cost"),
        fc(df_costs_raw, "ListCost", "double").alias("list_cost"),
        fc(df_costs_raw, "ContractedCost", "double").alias("contracted_cost"),
        fc(df_costs_raw, "x_BilledCostInUsd", "double").alias("billed_cost_usd"),
        fc(df_costs_raw, "ConsumedQuantity", "double").alias("consumed_quantity"),
        fc(df_costs_raw, "PricingQuantity", "double").alias("pricing_quantity"),
    )
    .withColumn("is_ai_service", (F.col("service_category") == F.lit("AI and Machine Learning")))
    .withColumn("is_foundry_models", (F.col("meter_category") == F.lit("Foundry Models")))
    .withColumn("cost_year", F.year("charge_date"))
    .withColumn("cost_month", F.date_format("charge_date", "yyyy-MM"))
)

write_table(fact_cost, "fact_cost")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 8, Finished, Available, Finished, False)

  wrote analytics.fact_cost: 11259 rows


## 4. `fact_genai_calls`
One row per gen_ai span (`chat`, `invoke_agent`, `execute_tool`, `invoke_tool`) from AppDependencies.
Consolidates **usage, tokens, latency, errors and tool calls** into a single fact so the model stays lean.

In [7]:
GEN_AI_SPANS = ["chat", "invoke_agent", "execute_tool", "invoke_tool"]

base = df_appdeps_raw.filter(
    (F.col("Type") == "AppDependencies") & (F.col("Data").isin(GEN_AI_SPANS))
)

fact_genai_calls = (
    base
    .withColumn("event_time", F.to_timestamp("TimeGenerated"))
    .select(
        F.col("event_time"),
        F.to_date("event_time").alias("date_key"),
        F.col("Data").alias("span_type"),
        sc(base, "Properties", "gen_ai.operation.name").alias("operation_name"),
        sc(base, "Properties", "gen_ai.conversation.id").alias("conversation_id"),
        sc(base, "Properties", "gen_ai.response.id").alias("response_id"),
        sc(base, "Properties", "gen_ai.agent.id").alias("agent_id"),
        sc(base, "Properties", "gen_ai.agent.name").alias("agent_name"),
        sc(base, "Properties", "gen_ai.agent.version").alias("agent_version"),
        sc(base, "Properties", "microsoft.foundry.project.id").alias("project_id"),
        sc(base, "Properties", "gen_ai.request.model").alias("request_model"),
        sc(base, "Properties", "gen_ai.response.model").alias("response_model"),
        F.coalesce(
            sc(base, "Properties", "gen_ai.provider.name"),
            sc(base, "Properties", "gen_ai.system"),
        ).alias("provider"),
        sc(base, "Properties", "gen_ai.response.finish_reasons").alias("finish_reasons"),
        sc(base, "Properties", "microsoft.foundry.content_filter.results").alias("content_filter_results"),
        sc(base, "Properties", "microsoft.a365.agent.blueprint.id").alias("blueprint_id"),
        sc(base, "Properties", "gen_ai.tool.name").alias("tool_name"),
        sc(base, "Properties", "gen_ai.tool.type").alias("tool_type"),
        fc(base, "DurationMs", "double").alias("duration_ms"),
        fc(base, "Success", "boolean").alias("success"),
        fc(base, "ResultCode", "string").alias("result_code"),
        fc(base, "ItemCount", "long").alias("item_count"),
        F.coalesce(sc(base, "Measurements", "gen_ai.usage.input_tokens", "double"), F.lit(0)).cast("long").alias("input_tokens"),
        F.coalesce(sc(base, "Measurements", "gen_ai.usage.output_tokens", "double"), F.lit(0)).cast("long").alias("output_tokens"),
        F.coalesce(sc(base, "Measurements", "gen_ai.usage.cached_tokens", "double"), F.lit(0)).cast("long").alias("cached_tokens"),
    )
    .withColumn("total_tokens", F.col("input_tokens") + F.col("output_tokens"))
    .withColumn("foundry_account", F.regexp_extract(F.coalesce(F.col("project_id"), F.lit("")), r"/accounts/([^/]+)", 1))
    .withColumn("foundry_project", F.regexp_extract(F.coalesce(F.col("project_id"), F.lit("")), r"/projects/([^/]+)", 1))
    .withColumn("is_error", (~F.coalesce(F.col("success"), F.lit(False))))
)

write_table(fact_genai_calls, "fact_genai_calls")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 9, Finished, Available, Finished, False)

  wrote analytics.fact_genai_calls: 52 rows


## 5. `fact_token_consumption` (report page: token usage)
Token totals per conversation x agent x model x project x day (chat spans only). Feeds the token-consumption
report page. User identity is excluded (not present in the exported dataset).

In [8]:
fact_token_consumption = (
    fact_genai_calls
    .filter(F.col("span_type") == "chat")
    .groupBy(
        F.col("date_key").alias("timeday"),
        "conversation_id",
        "agent_id",
        "agent_name",
        "response_model",
        "project_id",
        "foundry_account",
        "foundry_project",
    )
    .agg(
        F.max("event_time").alias("last_event_time"),
        F.sum("cached_tokens").alias("sum_cached_tokens"),
        F.sum("input_tokens").alias("sum_input_tokens"),
        F.sum("output_tokens").alias("sum_output_tokens"),
        F.sum("total_tokens").alias("sum_total_tokens"),
        F.countDistinct("response_id").alias("response_count"),
    )
)

write_table(fact_token_consumption, "fact_token_consumption")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 10, Finished, Available, Finished, False)

  wrote analytics.fact_token_consumption: 7 rows


## 6. `fact_conversation_steps` (report page: conversation explorer)
Ordered steps of each agent conversation across `chat`, `execute_tool` and `invoke_tool` spans, including
the input/output messages and tool-call payloads. `step_number` orders the steps within a conversation by
time. Feedback and user identity are intentionally excluded.

In [9]:
STEP_SPANS = ["chat", "execute_tool", "invoke_tool"]

steps_base = (
    df_appdeps_raw
    .filter((F.col("Type") == "AppDependencies") & (F.col("Data").isin(STEP_SPANS)))
    .withColumn("event_time", F.to_timestamp("TimeGenerated"))
)

steps_sel = (
    steps_base.select(
        F.col("event_time"),
        F.to_date("event_time").alias("date_key"),
        F.col("Data").alias("span_type"),
        sc(steps_base, "Properties", "gen_ai.conversation.id").alias("conversation_id"),
        sc(steps_base, "Properties", "gen_ai.response.id").alias("response_id"),
        sc(steps_base, "Properties", "gen_ai.operation.name").alias("operation_name"),
        sc(steps_base, "Properties", "gen_ai.agent.id").alias("agent_id"),
        sc(steps_base, "Properties", "gen_ai.agent.name").alias("agent_name"),
        sc(steps_base, "Properties", "gen_ai.agent.version").alias("agent_version"),
        sc(steps_base, "Properties", "gen_ai.response.model").alias("response_model"),
        sc(steps_base, "Properties", "microsoft.foundry.project.id").alias("project_id"),
        sc(steps_base, "Properties", "gen_ai.input.messages").alias("input_messages"),
        sc(steps_base, "Properties", "gen_ai.output.messages").alias("output_messages"),
        sc(steps_base, "Properties", "gen_ai.tool.name").alias("tool_name"),
        sc(steps_base, "Properties", "gen_ai.tool.type").alias("tool_type"),
        sc(steps_base, "Properties", "gen_ai.tool.call.arguments").alias("tool_call_arguments"),
        sc(steps_base, "Properties", "gen_ai.tool.call.result").alias("tool_call_result"),
        sc(steps_base, "Properties", "gen_ai.tool.definitions").alias("tool_definitions"),
        sc(steps_base, "Properties", "gen_ai.response.finish_reasons").alias("finish_reasons"),
        sc(steps_base, "Properties", "microsoft.foundry.content_filter.results").alias("content_filter_results"),
        fc(steps_base, "DurationMs", "double").alias("duration_ms"),
        fc(steps_base, "Success", "boolean").alias("success"),
        fc(steps_base, "ResultCode", "string").alias("result_code"),
        F.coalesce(sc(steps_base, "Measurements", "gen_ai.usage.input_tokens", "double"), F.lit(0)).cast("long").alias("input_tokens"),
        F.coalesce(sc(steps_base, "Measurements", "gen_ai.usage.output_tokens", "double"), F.lit(0)).cast("long").alias("output_tokens"),
        F.coalesce(sc(steps_base, "Measurements", "gen_ai.usage.cached_tokens", "double"), F.lit(0)).cast("long").alias("cached_tokens"),
    )
    .withColumn("foundry_account", F.regexp_extract(F.coalesce(F.col("project_id"), F.lit("")), r"/accounts/([^/]+)", 1))
    .withColumn("foundry_project", F.regexp_extract(F.coalesce(F.col("project_id"), F.lit("")), r"/projects/([^/]+)", 1))
    .withColumn("total_tokens", F.col("input_tokens") + F.col("output_tokens"))
)

step_w = Window.partitionBy("conversation_id").orderBy("event_time", "response_id")
fact_conversation_steps = steps_sel.withColumn("step_number", F.row_number().over(step_w))

write_table(fact_conversation_steps, "fact_conversation_steps")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 11, Finished, Available, Finished, False)

  wrote analytics.fact_conversation_steps: 28 rows


## 7. `dim_agent` & `dim_model`
Small conformed dimensions derived from the gen_ai fact for clean Power BI slicers.

In [10]:
dim_agent = (
    fact_genai_calls
    .filter(F.col("agent_id").isNotNull())
    .groupBy("agent_id")
    .agg(
        F.max("agent_name").alias("agent_name"),
        F.max("agent_version").alias("agent_version"),
        F.max("blueprint_id").alias("blueprint_id"),
        F.max("foundry_account").alias("foundry_account"),
        F.max("foundry_project").alias("foundry_project"),
        F.max("project_id").alias("project_id"),
    )
)
write_table(dim_agent, "dim_agent")

dim_model = (
    fact_genai_calls
    .filter(F.col("response_model").isNotNull())
    .groupBy(F.col("response_model").alias("model_name"))
    .agg(
        F.max("provider").alias("provider"),
        F.max("request_model").alias("request_model"),
    )
)
write_table(dim_model, "dim_model")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 12, Finished, Available, Finished, False)

  wrote analytics.dim_agent: 3 rows
  wrote analytics.dim_model: 2 rows


## 7b. `dim_conversation`
Conformed conversation dimension (unique `conversation_id`) so token consumption and conversation steps
share a conversation key — this enables drill-through filtering by conversation (and agent).

In [11]:
conv_cols = ["conversation_id", "agent_id", "agent_name", "response_model", "foundry_account", "foundry_project"]

dim_conversation = (
    fact_conversation_steps.select(*conv_cols)
    .unionByName(fact_token_consumption.select(*conv_cols), allowMissingColumns=True)
    .unionByName(fact_genai_calls.select(*conv_cols), allowMissingColumns=True)
    .filter(F.col("conversation_id").isNotNull())
    .groupBy("conversation_id")
    .agg(
        F.max("agent_id").alias("agent_id"),
        F.max("agent_name").alias("agent_name"),
        F.max("response_model").alias("response_model"),
        F.max("foundry_account").alias("foundry_account"),
        F.max("foundry_project").alias("foundry_project"),
    )
)
write_table(dim_conversation, "dim_conversation")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 13, Finished, Available, Finished, False)

  wrote analytics.dim_conversation: 5 rows


## 8. `dim_resource`
Resource inventory + tags from Azure Resource Graph, deduplicated to the latest snapshot, with governance
flags (owner/project/environment tags) and an AI-resource flag. Joins to `fact_cost[resource_id]`.

In [12]:
meta = df_metadata_raw.filter(fc(df_metadata_raw, "id").isNotNull())

w = Window.partitionBy(F.col("`id`")).orderBy(F.col("`_source_file`").desc())
meta_latest = (
    meta.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
)

dim_resource = (
    meta_latest.select(
        fc(meta_latest, "id").alias("resource_id"),
        fc(meta_latest, "name").alias("resource_name"),
        fc(meta_latest, "type").alias("resource_type"),
        fc(meta_latest, "location").alias("location"),
        fc(meta_latest, "resourceGroup").alias("resource_group"),
        fc(meta_latest, "subscriptionId").alias("subscription_id"),
        fc(meta_latest, "kind").alias("kind"),
        fc(meta_latest, "sku.name").alias("sku_name"),
        fc(meta_latest, "sku.tier").alias("sku_tier"),
        fc(meta_latest, "sku.family").alias("sku_family"),
        fc(meta_latest, "sku.size").alias("sku_size"),
        fc(meta_latest, "identity.type").alias("identity_type"),
        fc(meta_latest, "identity.principalId").alias("identity_principal_id"),
        fc(meta_latest, "tags.environment").alias("tag_environment"),
        fc(meta_latest, "tags.project").alias("tag_project"),
        fc(meta_latest, "tags.azd-env-name").alias("tag_azd_env"),
        fc(meta_latest, "tags.azd-service-name").alias("tag_azd_service"),
        fc(meta_latest, "tags.Contact").alias("tag_contact"),
        fc(meta_latest, "tags.CreatedByPolicy").alias("tag_created_by_policy"),
        fc(meta_latest, "tags.securityControl").alias("tag_security_control"),
    )
    .withColumn("is_ai_resource",
        F.lower(F.col("resource_type")).rlike("cognitiveservices|search/searchservices|botservice|machinelearningservices"))
    .withColumn("has_owner_tag", F.col("tag_contact").isNotNull())
    .withColumn("has_project_tag", F.col("tag_project").isNotNull())
    .withColumn("has_environment_tag", F.col("tag_environment").isNotNull())
    .withColumn("is_governed",
        F.col("tag_contact").isNotNull() & F.col("tag_project").isNotNull() & F.col("tag_environment").isNotNull())
)

write_table(dim_resource, "dim_resource")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 14, Finished, Available, Finished, False)

  wrote analytics.dim_resource: 139 rows


## 9. `dim_date`
Contiguous date dimension covering the years present in the fact tables.

In [13]:
bounds = (
    fact_cost.select(F.col("charge_date").alias("d"))
    .union(fact_genai_calls.select(F.col("date_key").alias("d")))
    .agg(F.min("d").alias("mn"), F.max("d").alias("mx"))
    .collect()[0]
)
min_d = bounds["mn"] or date(2026, 1, 1)
max_d = bounds["mx"] or date(2026, 12, 31)
start = date(min_d.year, 1, 1)
end = date(max_d.year, 12, 31)

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{start}'), to_date('{end}'), interval 1 day)) AS date_key")
    .withColumn("date_int", F.date_format("date_key", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date_key"))
    .withColumn("quarter", F.quarter("date_key"))
    .withColumn("month", F.month("date_key"))
    .withColumn("month_name", F.date_format("date_key", "MMMM"))
    .withColumn("month_short", F.date_format("date_key", "MMM"))
    .withColumn("year_month", F.date_format("date_key", "yyyy-MM"))
    .withColumn("year_quarter", F.concat_ws("-Q", F.year("date_key").cast("string"), F.quarter("date_key").cast("string")))
    .withColumn("month_start", F.trunc("date_key", "MM"))
    .withColumn("day", F.dayofmonth("date_key"))
    .withColumn("day_of_week", F.dayofweek("date_key"))
    .withColumn("day_name", F.date_format("date_key", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("date_key"))
    .withColumn("is_weekend", F.dayofweek("date_key").isin(1, 7))
)

write_table(dim_date, "dim_date")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 15, Finished, Available, Finished, False)

  wrote analytics.dim_date: 365 rows


## 10. Validation

In [14]:
for t in ["dim_date", "dim_resource", "dim_agent", "dim_model", "dim_conversation", "fact_cost", "fact_genai_calls", "fact_token_consumption", "fact_conversation_steps"]:
    cnt = spark.read.format("delta").load(f"Tables/{SCHEMA}/{t}").count()
    print(f"{t}: {cnt} rows")


StatementMeta(, 530a0571-e331-4998-8e94-91ba2d522a74, 16, Finished, Available, Finished, False)

dim_date: 365 rows
dim_resource: 139 rows
dim_agent: 3 rows
dim_model: 2 rows
dim_conversation: 5 rows
fact_cost: 11259 rows
fact_genai_calls: 52 rows
fact_token_consumption: 7 rows
fact_conversation_steps: 28 rows
